In [24]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [32]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test  = pd.read_csv("../data/processed/X_test.csv")

y_cost_train = pd.read_csv("../data/processed/y_cost_train.csv").squeeze()
y_cost_test  = pd.read_csv("../data/processed/y_cost_test.csv").squeeze()

print(X_train.shape, X_test.shape)


(2080, 12) (520, 12)


In [33]:
# Data Quality Check (No removal - keep all data for training)
print("=" * 50)
print("Data Quality Analysis")
print("=" * 50)

# Calculate IQR for y_cost_train
Q1 = y_cost_train.quantile(0.25)
Q3 = y_cost_train.quantile(0.75)
IQR = Q3 - Q1

# Define outlier boundaries
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1: {Q1:.4f}, Q3: {Q3:.4f}, IQR: {IQR:.4f}")
print(f"Lower Bound: {lower_bound:.4f}, Upper Bound: {upper_bound:.4f}")
outlier_count = ((y_cost_train < lower_bound) | (y_cost_train > upper_bound)).sum()
print(f"Outliers detected: {outlier_count} ({(outlier_count/len(y_cost_train)*100):.2f}%)")
print(f"Total training samples: {len(y_cost_train)}")
print("→ Keeping all data for better generalization with feature engineering")
print("=" * 50)


Data Quality Analysis
Q1: 0.2000, Q3: 0.3500, IQR: 0.1500
Lower Bound: -0.0250, Upper Bound: 0.5750
Outliers detected: 96 (4.62%)
Total training samples: 2080
→ Keeping all data for better generalization with feature engineering


In [ ]:
# Feature Engineering - Create new derived features
print("=" * 50)
print("Feature Engineering")
print("=" * 50)

X_train_fe = X_train.copy()
X_test_fe = X_test.copy()

# Define numeric and categorical columns
numeric_cols = X_train_fe.select_dtypes(include=[np.number]).columns.tolist()

# 1. Interaction features
if 'strength' in numeric_cols and 'weight_capacity' in numeric_cols:
    X_train_fe['strength_weight_product'] = X_train_fe['strength'] * X_train_fe['weight_capacity']
    X_test_fe['strength_weight_product'] = X_test_fe['strength'] * X_test_fe['weight_capacity']
    
    # Normalized interaction
    strength_max = X_train_fe['strength'].max()
    weight_max = X_train_fe['weight_capacity'].max()
    X_train_fe['strength_weight_normalized'] = (X_train_fe['strength'] / (strength_max + 1e-6)) * (X_train_fe['weight_capacity'] / (weight_max + 1e-6))
    X_test_fe['strength_weight_normalized'] = (X_test_fe['strength'] / (strength_max + 1e-6)) * (X_test_fe['weight_capacity'] / (weight_max + 1e-6))
    
if 'biodegradability_score' in numeric_cols and 'recyclability_percentage' in numeric_cols:
    X_train_fe['eco_quality_score'] = X_train_fe['biodegradability_score'] * X_train_fe['recyclability_percentage']
    X_test_fe['eco_quality_score'] = X_test_fe['biodegradability_score'] * X_test_fe['recyclability_percentage']

# 2. Ratio features (cost efficiency)
if 'strength' in numeric_cols:
    X_train_fe['strength_ratio'] = X_train_fe['strength'] / (X_train_fe['strength'].max() + 1e-6)
    X_test_fe['strength_ratio'] = X_test_fe['strength'] / (X_train_fe['strength'].max() + 1e-6)
    
if 'weight_capacity' in numeric_cols:
    X_train_fe['weight_capacity_ratio'] = X_train_fe['weight_capacity'] / (X_train_fe['weight_capacity'].max() + 1e-6)
    X_test_fe['weight_capacity_ratio'] = X_test_fe['weight_capacity'] / (X_train_fe['weight_capacity'].max() + 1e-6)

# 3. Polynomial features (squared terms) 
if 'biodegradability_score' in numeric_cols:
    X_train_fe['biodegradability_squared'] = X_train_fe['biodegradability_score'] ** 2
    X_test_fe['biodegradability_squared'] = X_test_fe['biodegradability_score'] ** 2
    
    X_train_fe['biodegradability_cubed'] = X_train_fe['biodegradability_score'] ** 3
    X_test_fe['biodegradability_cubed'] = X_test_fe['biodegradability_score'] ** 3

if 'recyclability_percentage' in numeric_cols:
    X_train_fe['recyclability_squared'] = X_train_fe['recyclability_percentage'] ** 2
    X_test_fe['recyclability_squared'] = X_test_fe['recyclability_percentage'] ** 2

# 4. Material cost indices (weighted by material type if available)
material_cols = [col for col in X_train_fe.columns if col.startswith('material_type_')]
if material_cols:
    X_train_fe['material_diversity'] = X_train_fe[material_cols].sum(axis=1)
    X_test_fe['material_diversity'] = X_test_fe[material_cols].sum(axis=1)

print(f"Original features: {X_train.shape[1]}")
print(f"After feature engineering: {X_train_fe.shape[1]}")
print(f"New features added: {X_train_fe.shape[1] - X_train.shape[1]}")
new_features = [col for col in X_train_fe.columns if col not in X_train.columns]
print(f"New feature columns: {new_features}")
print("=" * 50)

# Update X_train and X_test
X_train = X_train_fe
X_test = X_test_fe


Feature Engineering
Numeric columns: ['strength', 'weight_capacity', 'biodegradability_score', 'recyclability_percentage', 'fragility_level']
Original features: 12
After feature engineering: 18
New features added: 6
New feature columns: ['strength_weight_product', 'eco_quality', 'cost_per_strength', 'weight_per_cost_ratio', 'biodegradability_squared', 'recyclability_squared']


In [ ]:
print("Initializing Enhanced Random Forest Cost Model...")
print("Hyperparameters:")
print("- n_estimators: 500 (more trees for better fit)")
print("- max_depth: 18 (increased for complex relationships)")
print("- min_samples_split: 4 (encourage more splits)")
print("- min_samples_leaf: 1 (capture patterns)")
print("- max_features: sqrt (feature subsampling)")
print("=" * 50)

rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=18,
    min_samples_split=4,
    min_samples_leaf=1,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1,
    verbose=0
)


In [36]:
print("Training Random Forest Cost Model...")
rf_model.fit(X_train, y_cost_train)
print("✓ Training complete")

Training Random Forest Cost Model...
✓ Training complete


In [37]:
y_cost_pred = rf_model.predict(X_test)


In [38]:
rmse = np.sqrt(mean_squared_error(y_cost_test, y_cost_pred))
mae  = mean_absolute_error(y_cost_test, y_cost_pred)
r2   = r2_score(y_cost_test, y_cost_pred)

print("=" * 50)
print("Random Forest Cost Prediction Metrics")
print("=" * 50)
print(f"RMSE: {rmse:.4f}")
print(f"MAE : {mae:.4f}")
print(f"R²  : {r2:.4f}")
print("=" * 50)

Random Forest Cost Prediction Metrics
RMSE: 0.0760
MAE : 0.0570
R²  : 0.6665


In [15]:
feature_importance = pd.Series(
    rf_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

feature_importance.head(10)


biodegradability_score      0.737017
recyclability_percentage    0.073404
weight_capacity             0.063419
strength                    0.053853
material_type_glass         0.020965
material_type_bamboo        0.013326
fragility_level             0.012128
material_type_metal         0.011512
material_type_paper         0.005530
material_type_jute          0.004803
dtype: float64

In [23]:
import joblib

joblib.dump(rf_model, "../models/rf_cost_model.pkl")


['../models/rf_cost_model.pkl']